In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid');

Отступить на epsilon от границы дырок.

In [3]:
%pwd

'C:\\Users\\CompAdmin\\Desktop\\dqn\\hse\\LAB\\TDA\\language_holes_and_voids\\adhocs'

# RU

## words

In [4]:
filtered_by_syn = np.load("../../EXP/stb-tda/holes/RU/words/filtered_by_lifetimes_and_syn.npy", allow_pickle=True).item()

In [5]:
words = np.load('../../EXP/word_cbow.npy')
word_partition = np.load('../../EXP/stb-tda/split_space_kmeans/RU/words/ru_word_partition.npy')

In [10]:
import sys
sys.path.append("../../EXP/stb-tda/src")
sys.path.append("../../EXP/stb-tda/")
from src.holes_utils import get_cycles

In [11]:
import glob
def get_holes_per_chunk(dir="holes/RU/words", filenames="ru_word_holes_*.npy"):
    hole_contours = {}
    for file in glob.glob(f"{dir}/{filenames}"):
        hole_contours["_".join(file.split('_')[3:])[:-4]] = get_cycles(np.load(file) - 1)
    return hole_contours
h1_contours = get_holes_per_chunk("../../EXP/stb-tda/holes/RU/words", "ru_word_holes_*.npy")

In [6]:
filtered_by_syn

{'0': array([488], dtype=int64),
 '1111': array([400, 415, 435, 444, 470, 492], dtype=int64)}

In [12]:
for c in filtered_by_syn:
    cluster_idx = np.where(word_partition == c)[0]
    print(c)
    print('*' * 10)
    for hole_num in filtered_by_syn[c]:
        print(words[cluster_idx[h1_contours[c][hole_num]]].tolist())
    print('\n')

0
**********
['смешливый', 'черноглазый', 'чернобровый', 'молодица', 'баять', 'вишь', 'сякой', 'срамить', 'бранить', 'сердиться', 'конфузиться', 'сконфузить', 'смущенный', 'растерянный', 'озадаченный', 'ошарашивать', 'оторопеть', 'озадаченно', 'многозначительно', 'благожелательно', 'доброжелательно', 'дружелюбный', 'приветливый', 'добродушный', 'простоватый', 'разбитной', 'бойкий']


1111
**********
['супить', 'вздергивание', 'зиновия', 'езерский', 'археографический', 'поштучный', 'лжесвидетельствовать', 'топотня', 'ржавчинка', 'хрипливый', 'насморочный', 'капок']
['препаровочный', 'подзеркальный', 'пеленальный', 'афины', 'переворачивание', 'семушка', 'юрат', 'собачина', 'прасольский', 'акселерометр']
['орочский', 'полотерный', 'кочедыжник', 'текстолит', 'застынуть', 'ахея', 'златогривый', 'расковываться', 'выпрягаться']
['строгание', 'гидрат', 'текстолит', 'застынуть', 'ерон', 'прорезывание', 'пломбировать', 'пилиться']
['предсмертно', 'загород', 'мезонинный', 'орджоникидзевский', 'бу

In [59]:
lang = 'ru'
part = 'word'
eps = 5e-3
for c in filtered_by_syn:
    cluster_idx = np.where(word_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    print(c)
    print('*' * 10)
    # cycle_idx_list = cluster_idx[h1_contours[c][hole_num]]
    for hole_num in filtered_by_syn[c]:
        print(words[cluster_idx[h1_contours[c][hole_num]]].tolist())

        hole_emb = chunk_emb[h1_contours[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        print("+eps")
        print(words[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        print("\n")
    print('\n')

0
**********
['смешливый', 'черноглазый', 'чернобровый', 'молодица', 'баять', 'вишь', 'сякой', 'срамить', 'бранить', 'сердиться', 'конфузиться', 'сконфузить', 'смущенный', 'растерянный', 'озадаченный', 'ошарашивать', 'оторопеть', 'озадаченно', 'многозначительно', 'благожелательно', 'доброжелательно', 'дружелюбный', 'приветливый', 'добродушный', 'простоватый', 'разбитной', 'бойкий']
+eps
['адъютант', 'благодетельница', 'буян', 'возражать', 'волноваться', 'встрепенуться', 'городовой', 'дита', 'донна', 'замешательство', 'запинка', 'засуетиться', 'игуменья', 'ильич', 'нашкодить', 'недовольно', 'недостойно', 'немножко', 'неожиданно', 'нечисто', 'отощать', 'перемолвиться', 'полицеймейстер', 'понимающе', 'потрясенно', 'привечать', 'прохвост', 'ровесник', 'русак', 'свадебка', 'скоморох', 'солдатик', 'спохватываться', 'тактично', 'тамбовский', 'телеграфист', 'трезвый', 'третировать', 'удумывать', 'униматься', 'форейтор', 'хамить', 'хозяйка', 'шибко', 'шуточка', 'этак', 'ямщик', 'ярославский']



In [88]:
import json

In [91]:
lang = 'ru'
part = 'word'
eps = 5e-3

w_contours = []
for c in filtered_by_syn:
    cluster_idx = np.where(word_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    # print(c)
    # print('*' * 10)
    # cycle_idx_list = cluster_idx[h1_contours[c][hole_num]]
    for i, hole_num in enumerate(filtered_by_syn[c]):
        contour_words = words[cluster_idx[h1_contours[c][hole_num]]].tolist()
        # print(words[cluster_idx[h1_contours[c][hole_num]]].tolist())

        hole_emb = chunk_emb[h1_contours[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        # print("+eps")
        # print(words[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        eps_words = words[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist()
        w_contours.append({"contour": contour_words, "contour + eps": eps_words})
        # print("\n")
    # print('\n')

In [93]:
import json
with open("ru_word_contour_eps.json", "w") as f:
    json.dump(w_contours, f)

## bigrams

In [60]:
bigrams = np.load("../../EXP/semantic_space/ru_bigrams_100k.npy")
bigram_partition = np.load('../../EXP/stb-tda/split_space_kmeans/RU/bigrams/ru_bigram_100k_partition_new.npy')
np.unique(bigram_partition)

array(['0_0', '0_1', '10_0', '10_1', '11', '12', '13', '14_0', '14_1',
       '15', '16', '1_0', '1_1', '2', '3', '4_0', '4_1', '5', '6', '7',
       '8', '9'], dtype='<U11')

In [66]:
filtered_by_hole_diam_bi = np.load("../../EXP/stb-tda/holes/RU/bigrams/filtered_by_lifetimes_and_hole_diam.npy", allow_pickle=True).item()

In [76]:
h1_contours_bi = get_holes_per_chunk("../../EXP/stb-tda/holes/RU/bigrams/", "ru_bigram_holes_*.npy")

In [79]:
lang = 'ru'
part = 'bigram'

for c in filtered_by_hole_diam_bi:
    cluster_idx = np.where(bigram_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    
    print(c)
    print('*' * 10)
    for hole_num in filtered_by_hole_diam_bi[c]:
        print(bigrams[cluster_idx[h1_contours_bi[c][hole_num]]].tolist())

        hole_emb = chunk_emb[h1_contours_bi[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        print("+eps")
        print(bigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        print("\n")
    print('\n')

13
**********
['кольчуга рубаха', 'простецкий рубаха', 'винтаж куртка', 'флотский комбинезон', 'полетный скафандр', 'суставный капсула', 'волок контейнер', 'приставной емкость', 'грязноватый емкость', 'хон бутыль', 'притворщик банка', 'извлекать банка', 'извлекать аккуратно', 'пробирка аккуратно', 'прогундосить аккуратно', 'жених аккуратно', 'жених колено', 'муж нога', 'обминаться нога', 'утяжелитель нога', 'сражение нога', 'сражение плечо', 'диагональный плечо', 'отражаться плечо', 'отражаться щека', 'светодиод щека', 'спичка щека', 'свечка лоб', 'вдариться лоб', 'гладить лоб', 'поглаживать шея', 'кутенок шея', 'проходить шея', 'пройти кожа', 'кольчуга кожа', 'плавить кожа']
+eps
['поверх лицо', 'припирать зуб', 'щипцы ножницы', 'кавалерийский пистолет', 'компактный диск', 'вечер сумка', 'перетягивать скатерть', 'перемучиться тушка', 'тереть дощечка', 'эффект челюсть', 'священный крыло', 'единственный плитка', 'прок клинок', 'туман юбка', 'прикалывать зеркало', 'вернуться стакан', 'же

In [94]:
lang = 'ru'
part = 'bigram'

b_contours = []
for c in filtered_by_hole_diam_bi:
    cluster_idx = np.where(bigram_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    
    # print(c)
    # print('*' * 10)
    for hole_num in filtered_by_hole_diam_bi[c]:
        # print(bigrams[cluster_idx[h1_contours_bi[c][hole_num]]].tolist())
        bigram_contours = bigrams[cluster_idx[h1_contours_bi[c][hole_num]]].tolist()

        hole_emb = chunk_emb[h1_contours_bi[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        # print("+eps")
        # print(bigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        bigram_eps_contours = bigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist()
        b_contours.append({'contour': bigram_contours, 'contour + eps': bigram_eps_contours})
        # print("\n")
    # print('\n')

In [95]:
with open("ru_bigram_contour_eps.json", "w") as f:
    json.dump(b_contours, f)

## trigrams

In [81]:
trigrams = np.load("../../EXP/semantic_space/ru_trigrams_100k.npy")
trigram_partition = np.load('../../EXP/stb-tda/split_space_kmeans/RU/trigrams/ru_trigram_partition_new.npy')
np.unique(trigram_partition)

array(['00', '01', '02', '1', '10', '11', '12', '13', '14', '15', '160',
       '161', '162', '17', '18', '19', '2', '20', '210', '211', '212',
       '22', '230', '231', '232', '240', '241', '242', '250', '251',
       '252', '253', '254', '260', '261', '262', '27', '28', '29', '3',
       '4', '5', '60', '61', '62', '70', '71', '72', '80', '81', '82',
       '90', '91', '92', '93', '94'], dtype='<U11')

In [82]:
filtered_by_hole_diam_tri = np.load("../../EXP/stb-tda/holes/RU/trigrams/filtered_by_lifetimes_and_hole_diam.npy", allow_pickle=True).item()
h1_contours_tri = get_holes_per_chunk("../../EXP/stb-tda/holes/RU/trigrams/", "ru_trigram_holes_*.npy")

In [86]:
lang = 'ru'
part = 'trigram'

for c in filtered_by_hole_diam_tri:
    cluster_idx = np.where(trigram_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    
    print(c)
    print('*' * 10)
    for hole_num in filtered_by_hole_diam_tri[c]:
        print(trigrams[cluster_idx[h1_contours_tri[c][hole_num]]].tolist())

        hole_emb = chunk_emb[h1_contours_tri[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        print("+eps")
        print(trigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        print("\n")
    print('\n')

210
**********
['домой нива свой', 'домой город наступление', 'домой командующий фронт', 'прозывать командующий фронт', 'бывший офицер офицер', 'вместе полк товарищ', 'свой отряд товарищ', 'свой ординарец оренбургский', 'свой нива полагаться', 'свой лионский землячка']
+eps
['камень солдат офицер', 'деревня холм офицер', 'целый войско вооруженный', 'город ретивый офицер', 'старик военный офицерский', 'контролер воинский отряд', 'греческий десант высаживаться']


['толстый компрометировать советский', 'косоворотка красноармеец советский', 'пролетарский революция советский', 'ваш революция который', 'ваш партия знать', 'ваш отряд знать', 'свой отряд товарищ', 'оренбургский казак товарищ', 'сообща человек товарищ', 'толстый человек товарищ', 'толстый грек болгарин']
+eps
['молодой артиллерист прапорщик', 'отдавать большевик гусар', 'позволять большевик эвакуировать', 'мало война солдат']


['бедный человек богатый', 'ограбить соседство богатый', 'мужик деревня богатый', 'финляндский дерев

In [98]:
lang = 'ru'
part = 'trigram'

t_contours = []
for c in filtered_by_hole_diam_tri:
    cluster_idx = np.where(trigram_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    
    # print(c)
    # print('*' * 10)
    for hole_num in filtered_by_hole_diam_tri[c]:
        # print(trigrams[cluster_idx[h1_contours_bi[c][hole_num]]].tolist())
        trigram_contours = trigrams[cluster_idx[h1_contours_tri[c][hole_num]]].tolist()

        hole_emb = chunk_emb[h1_contours_tri[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        # print("+eps")
        # print(trigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        trigram_eps_contours = trigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist()
        t_contours.append({'contour': trigram_contours, 'contour + eps': trigram_eps_contours})
        # print("\n")
    # print('\n')

In [99]:
with open("ru_trigram_contour_eps.json", "w") as f:
    json.dump(t_contours, f)

# EN

## words

In [101]:
filtered_by_hole_diam = np.load("../../EXP/stb-tda/holes/EN/words/filtered_by_lifetimes_and_hole_diam.npy", allow_pickle=True).item()

In [102]:
words = np.load('../../EXP/en_word_cbow.npy')
word_partition = np.load('../../EXP/stb-tda/split_space_kmeans/EN/words/en_word_partition.npy')

In [103]:
h1_contours = get_holes_per_chunk("../../EXP/stb-tda/holes/EN/words", "en_word_holes_*.npy")

In [108]:
c = '310'
cluster_idx = np.where(word_partition == c)[0]
chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')

In [111]:
hole_num = filtered_by_hole_diam[c][0]

In [112]:
hole_emb = chunk_emb[h1_contours[c][hole_num]]
hole_centroid = hole_emb.mean(axis=0)
distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
max_r = distances.max()
distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
print("+eps")
print(words[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())

+eps
[]


In [113]:
distances

array([0.9766042 , 1.0251718 , 1.00587108, ..., 0.97183195, 0.94397373,
       1.01547673])

In [114]:
max_r

0.49972254061402643

In [116]:
lang = 'en'
part = 'word'
eps = 5e-3
for c in filtered_by_hole_diam:
    cluster_idx = np.where(word_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    print(c)
    print('*' * 10)
    # cycle_idx_list = cluster_idx[h1_contours[c][hole_num]]
    for hole_num in filtered_by_hole_diam[c]:
        print(words[cluster_idx[h1_contours[c][hole_num]]].tolist())

        hole_emb = chunk_emb[h1_contours[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        print("+eps")
        print(words[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        print("\n")
    print('\n')

310
**********
['spank', 'spoilt', 'motherless', 'childless', 'bewail', 'repine', 'flinch', 'budge']
+eps
[]


['vaccinator', 'meliorist', 'bigwiggery', 'talkt', 'whil', 'forsake', 'befriend', 'gainsay', 'condone']
+eps
['abdiel', 'academically', 'addressee', 'agouhanna', 'anfossi', 'anson', 'appleby', 'araxe', 'arge', 'argonautica', 'asymmetry', 'awmous', 'balti', 'basketry', 'batayle', 'bayete', 'bhai', 'bicuspid', 'bitidde', 'blowitz', 'bonapartist', 'boufe', 'braxie', 'brixton', 'brownell', 'bruner', 'buonespoir', 'burleigh', 'calimara', 'candescent', 'caressively', 'caruther', 'catchment', 'cavortin', 'chaffee', 'chappel', 'chetneys', 'chewink', 'chickabiddy', 'chiefely', 'chitinous', 'christchurch', 'chromos', 'cleigh', 'cocker', 'colchos', 'compotator', 'copmanhurst', 'cornea', 'crèpe', 'dairymaid', 'damne', 'darry', 'dedans', 'diademe', 'didius', 'dogana', 'doolittle', 'dootie', 'doto', 'dummer', 'dutifulness', 'duyckinck', 'dynamically', 'elagabalus', 'elsewhither', 'empte', '

In [88]:
import json

In [117]:
lang = 'en'
part = 'word'
eps = 5e-3

w_contours = []
for c in filtered_by_hole_diam:
    cluster_idx = np.where(word_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    # print(c)
    # print('*' * 10)
    # cycle_idx_list = cluster_idx[h1_contours[c][hole_num]]
    for i, hole_num in enumerate(filtered_by_hole_diam[c]):
        contour_words = words[cluster_idx[h1_contours[c][hole_num]]].tolist()
        # print(words[cluster_idx[h1_contours[c][hole_num]]].tolist())

        hole_emb = chunk_emb[h1_contours[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        # print("+eps")
        # print(words[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        eps_words = words[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist()
        w_contours.append({"contour": contour_words, "contour + eps": eps_words})
        # print("\n")
    # print('\n')

In [118]:
import json
with open("en_word_contour_eps.json", "w") as f:
    json.dump(w_contours, f)

## bigrams

In [119]:
bigrams = np.load("../../EXP/semantic_space/en_bigrams_100k.npy")
bigram_partition = np.load('../../EXP/stb-tda/split_space_kmeans/EN/bigrams/en_bigram_100k_partition_new.npy')
np.unique(bigram_partition)

array(['00', '01', '10', '100', '101', '11', '110', '111', '120', '121',
       '122', '123', '130', '131', '132', '133', '14', '20', '21', '22',
       '23', '30', '31', '40', '41', '5', '6', '70', '71', '80', '81',
       '9'], dtype='<U11')

In [125]:
filtered_by_hole_diam_bi = np.load("../../EXP/stb-tda/holes/EN/bigrams/filtered_by_lifetimes_and_hole_diam.npy", allow_pickle=True).item()

In [121]:
h1_contours_bi = get_holes_per_chunk("../../EXP/stb-tda/holes/EN/bigrams/", "en_bigram_holes_*.npy")

In [126]:
lang = 'en'
part = 'bigram'

for c in filtered_by_hole_diam_bi:
    cluster_idx = np.where(bigram_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    
    print(c)
    print('*' * 10)
    for hole_num in filtered_by_hole_diam_bi[c]:
        print(bigrams[cluster_idx[h1_contours_bi[c][hole_num]]].tolist())

        hole_emb = chunk_emb[h1_contours_bi[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        print("+eps")
        print(bigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        print("\n")
    print('\n')

100
**********
['elect episcopal', 'appoint arbiter', 'appoint these', 'authorize these', 'worker these', 'worker whose', 'goldsmith whose', 'worthy whose', 'worthy bhaer', 'worthy shepherd', 'royal shepherd', 'royal cuirassier', 'royal seat', 'elect seat']
+eps
['russian valet', 'include browning', 'needy spendthrift', 'corporation international', 'rich parishioner', 'destine of', 'foreign notepaper', 'subscription bounty', 'respectable angling', 'famous ninth', 'swedish archive', 'retainer meanwhile', 'namely viz', 'parish denounce', 'local carpentier', 'siege these', 'private commentary', 'famous loyalty', 'acquainted these', 'sovereign fair', 'retired plumber', 'respective harangue', 'pastor indirectly', 'provincial congress']




101
**********
['much bachelor', 'much anglo', 'much greek', 'townsfolk greek', 'and latin', 'above latin', 'above list', 'stockholder list', 'certain list', 'certain magazine', 'iscot magazine', 'marginal manuscript', 'scrawled letter', 'mismanage receip

In [127]:
lang = 'en'
part = 'bigram'

b_contours = []
for c in filtered_by_hole_diam_bi:
    cluster_idx = np.where(bigram_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    
    # print(c)
    # print('*' * 10)
    for hole_num in filtered_by_hole_diam_bi[c]:
        # print(bigrams[cluster_idx[h1_contours_bi[c][hole_num]]].tolist())
        bigram_contours = bigrams[cluster_idx[h1_contours_bi[c][hole_num]]].tolist()

        hole_emb = chunk_emb[h1_contours_bi[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        # print("+eps")
        # print(bigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        bigram_eps_contours = bigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist()
        b_contours.append({'contour': bigram_contours, 'contour + eps': bigram_eps_contours})
        # print("\n")
    # print('\n')

In [128]:
with open("en_bigram_contour_eps.json", "w") as f:
    json.dump(b_contours, f)

## trigrams

In [129]:
trigrams = np.load("../../EXP/semantic_space/en_trigrams_100k.npy")
trigram_partition = np.load('../../EXP/stb-tda/split_space_kmeans/EN/trigrams/en_trigram_partition_new.npy')
np.unique(trigram_partition)

array(['0_0', '0_1', '0_2', '10', '11_0', '11_1', '11_2', '12', '13_0',
       '13_1', '13_2', '14_0', '14_1', '14_2', '15', '16', '17', '18',
       '19_0', '19_1', '19_2', '1_0', '1_1', '1_2', '1_3', '1_4', '20',
       '21', '22_0', '22_1', '22_2', '23', '24', '25', '26', '27', '28',
       '29', '2_0', '2_1', '2_2', '2_3', '2_4', '3', '4_0', '4_1', '4_2',
       '5_0', '5_1', '5_2', '6', '7_0', '7_1', '7_2', '8_0', '8_1', '8_2',
       '9'], dtype='<U11')

In [130]:
filtered_by_hole_diam_tri = np.load("../../EXP/stb-tda/holes/EN/trigrams/filtered_by_lifetimes_and_hole_diam.npy", allow_pickle=True).item()
h1_contours_tri = get_holes_per_chunk("../../EXP/stb-tda/holes/EN/trigrams/", "en_trigram_holes_*.npy")

In [131]:
lang = 'en'
part = 'trigram'

for c in filtered_by_hole_diam_tri:
    cluster_idx = np.where(trigram_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    
    print(c)
    print('*' * 10)
    for hole_num in filtered_by_hole_diam_tri[c]:
        print(trigrams[cluster_idx[h1_contours_tri[c][hole_num]]].tolist())

        hole_emb = chunk_emb[h1_contours_tri[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        print("+eps")
        print(trigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        print("\n")
    print('\n')

7_0
**********
['storm be rage', 'blizzard be rage', 'face the rage', 'face unwashed and', 'face light with', 'moon light and', 'sun shine person1', 'sun so pron1', 'rain so that', 'wind so fragile', 'wind but still', 'wind would howl']
+eps
['weather also thyrsis', 'light pron1 whisper', 'chill pron1 delight', 'rain grow heavy']


['little old negress', 'half old baby', 'half insane and', 'half drunk and', 'almost drunk person1', 'almost spill pron1', 'honorific salad pron1', 'unimaginable viand pron1', 'unknowing unheeding pron1', 'little corydon pron1', 'little tawdry whore']
+eps
[]


['heart notwould be', 'heart ache person1', 'arm ache and', 'hand clench and', 'hand satchel and', 'hand pron1 face', 'kiss pron1 face', 'kiss pron1 with', 'smother pron1 with', 'thyrsis consecration with', 'boiling ledger with', 'cracker box with', 'soap box and', 'flour paste and', 'roast pork and', 'boil rice and', 'hot gravy and', 'hot blood spurt', 'heart blood and', 'cold blood pron1']
+eps
['co

In [132]:
lang = 'en'
part = 'trigram'

t_contours = []
for c in filtered_by_hole_diam_tri:
    cluster_idx = np.where(trigram_partition == c)[0]
    chunk_emb = np.load(f'../../EXP/stb-tda/split_space_kmeans/{lang.upper()}/{part}s/subsets/{lang}_{part}_{c}.npy')
    
    # print(c)
    # print('*' * 10)
    for hole_num in filtered_by_hole_diam_tri[c]:
        # print(trigrams[cluster_idx[h1_contours_bi[c][hole_num]]].tolist())
        trigram_contours = trigrams[cluster_idx[h1_contours_tri[c][hole_num]]].tolist()

        hole_emb = chunk_emb[h1_contours_tri[c][hole_num]]
        hole_centroid = hole_emb.mean(axis=0)
        distances = cdist(hole_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        max_r = distances.max()
        distances = cdist(chunk_emb, hole_centroid.reshape(1, -1), metric='cosine').flatten()
        # print("+eps")
        # print(trigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist())
        trigram_eps_contours = trigrams[cluster_idx[(distances > max_r) & (distances < max_r + eps)]].tolist()
        t_contours.append({'contour': trigram_contours, 'contour + eps': trigram_eps_contours})
        # print("\n")
    # print('\n')

In [133]:
with open("en_trigram_contour_eps.json", "w") as f:
    json.dump(t_contours, f)